Setup

In [0]:
%python
import os, time, zipfile
import pandas as pd

VOL = "/Volumes/workspace/default/airline_dataset"
ZIP_FILE = f"{VOL}/airline.csv.shuffle.zip"
CSV_FILE = f"{VOL}/airline.csv.shuffle"
RAW = "workspace.default.airline_raw"
CLEAN = "workspace.default.airline_clean"

RESULTS = []

def run(name, sql):
    t0 = time.perf_counter()
    out = spark.sql(sql).toPandas()
    RESULTS.append({"Stage": name, "Seconds": round(time.perf_counter() - t0, 3)})
    print(f"{name}  {RESULTS[-1]['Seconds']:,.2f} s")
    return out

Unzip and load file

In [0]:
%python
BENCHMARK_GB = 1.0
BLOCK = 64 * 1024 * 1024

BENCH_FILE = CSV_FILE if os.path.exists(CSV_FILE) else ZIP_FILE

t0 = time.perf_counter()

limit_bytes = int(min(BENCHMARK_GB * 1024 ** 3, os.path.getsize(BENCH_FILE)))
read_bytes = 0
with open(BENCH_FILE, "rb") as fh:
    while read_bytes < limit_bytes:
        block = fh.read(min(BLOCK, limit_bytes - read_bytes))
        if not block:
            break
        read_bytes += len(block)

RESULTS.append({"Stage": "0. Disk read benchmark (raw bytes)", "Seconds": round(time.perf_counter() - t0, 3)})
print(f"{read_bytes / 1024 ** 3:.2f} GB read from {os.path.basename(BENCH_FILE)} in {RESULTS[-1]['Seconds']:,.2f} s")

1.00 GB read from airline.csv.shuffle.zip in 1.40 s


In [0]:
%python
t0 = time.perf_counter()

if not os.path.exists(CSV_FILE):
    with zipfile.ZipFile(ZIP_FILE) as z:
        inner = [n for n in z.namelist() if not n.endswith("/")][0]
        with z.open(inner) as src, open(CSV_FILE, "wb") as dst:
            while True:
                block = src.read(64 * 1024 * 1024)
                if not block:
                    break
                dst.write(block)

spark.sql(f"""
CREATE OR REPLACE TABLE {RAW} AS
SELECT Year, Month, DayofMonth, DayOfWeek,
       UniqueCarrier, Origin, Dest, Distance,
       DepDelay, ArrDelay,
       Cancelled, Diverted, CancellationCode,
       CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay
FROM read_files(
    '{CSV_FILE}',
    format => 'csv',
    header => true,
    nullValue => 'NA',
    encoding => 'ISO-8859-1',
    schemaHints => 'Year INT, Month INT, DayofMonth INT, DayOfWeek INT,
                    UniqueCarrier STRING, Origin STRING, Dest STRING, CancellationCode STRING,
                    Distance DOUBLE, DepDelay DOUBLE, ArrDelay DOUBLE,
                    Cancelled DOUBLE, Diverted DOUBLE,
                    CarrierDelay DOUBLE, WeatherDelay DOUBLE, NASDelay DOUBLE,
                    SecurityDelay DOUBLE, LateAircraftDelay DOUBLE'
)
""")

RESULTS.append({"Stage": "1. Data loading", "Seconds": round(time.perf_counter() - t0, 3)})

n_raw = spark.sql(f"SELECT count(*) c FROM {RAW}").collect()[0]["c"]
print(f"{n_raw:,} rows loaded in {RESULTS[-1]['Seconds']:,.2f} s")

123,534,969 rows loaded in 325.26 s


In [0]:
%python
audit_wide = run("2a. Data quality audit (pre-clean)", f"""
SELECT count(*) AS n_rows,
       count(Year) AS Year, count(Month) AS Month,
       count(DayofMonth) AS DayofMonth, count(DayOfWeek) AS DayOfWeek,
       count(UniqueCarrier) AS UniqueCarrier, count(Origin) AS Origin, count(Dest) AS Dest,
       count(Distance) AS Distance, count(DepDelay) AS DepDelay, count(ArrDelay) AS ArrDelay,
       count(Cancelled) AS Cancelled, count(Diverted) AS Diverted,
       count(CancellationCode) AS CancellationCode,
       count(CarrierDelay) AS CarrierDelay, count(WeatherDelay) AS WeatherDelay,
       count(NASDelay) AS NASDelay, count(SecurityDelay) AS SecurityDelay,
       count(LateAircraftDelay) AS LateAircraftDelay
FROM {RAW}
""")

n_rows = int(audit_wide["n_rows"][0])
audit = audit_wide.drop(columns=["n_rows"]).T.reset_index()
audit.columns = ["Column", "Non_null"]
audit["Missing"] = n_rows - audit["Non_null"]
audit["Missing_pct"] = (100 * audit["Missing"] / n_rows).round(3)
display(audit.sort_values("Missing", ascending=False).reset_index(drop=True))

2a. Data quality audit (pre-clean)  1.33 s


Column,Non_null,Missing,Missing_pct
CancellationCode,734706,122800263,99.405
LateAircraftDelay,34205536,89329433,72.311
SecurityDelay,34205536,89329433,72.311
NASDelay,34205536,89329433,72.311
WeatherDelay,34205536,89329433,72.311
CarrierDelay,34205536,89329433,72.311
ArrDelay,120947440,2587529,2.095
DepDelay,121232833,2302136,1.864
Distance,123332969,202000,0.164
Month,123534969,0,0.0


In [0]:
%python
t0 = time.perf_counter()

spark.sql(f"""
CREATE OR REPLACE TABLE {CLEAN} AS
SELECT * EXCEPT (Cancelled, Diverted),
       (ArrDelay > 15) AS Delayed15
FROM (
    SELECT DISTINCT
           Year, Month, DayofMonth, DayOfWeek,
           UniqueCarrier, Origin, Dest, Distance,
           DepDelay, ArrDelay,
           Cancelled, Diverted, CancellationCode,
           CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay
    FROM {RAW}
    WHERE Year BETWEEN 1987 AND 2030
      AND Month BETWEEN 1 AND 12
      AND Cancelled = 0
      AND Diverted = 0
      AND ArrDelay IS NOT NULL
      AND abs(ArrDelay) <= 1440
      AND abs(coalesce(DepDelay, 0)) <= 1440
)
""")

RESULTS.append({"Stage": "2b. Cleaning", "Seconds": round(time.perf_counter() - t0, 3)})

n_clean = spark.sql(f"SELECT count(*) c FROM {CLEAN}").collect()[0]["c"]
print(f"{n_clean:,} rows kept in {RESULTS[-1]['Seconds']:,.2f} s")

118,503,331 rows kept in 37.90 s


In [0]:
%python
schema = run("3a. Schema description", f"""
WITH n AS (SELECT count(*) AS n_rows FROM {CLEAN}),
counts AS (
    SELECT 'Year' AS column_name, count(Year) AS non_null FROM {CLEAN}
    UNION ALL SELECT 'Month', count(Month) FROM {CLEAN}
    UNION ALL SELECT 'DayofMonth', count(DayofMonth) FROM {CLEAN}
    UNION ALL SELECT 'DayOfWeek', count(DayOfWeek) FROM {CLEAN}
    UNION ALL SELECT 'UniqueCarrier', count(UniqueCarrier) FROM {CLEAN}
    UNION ALL SELECT 'Origin', count(Origin) FROM {CLEAN}
    UNION ALL SELECT 'Dest', count(Dest) FROM {CLEAN}
    UNION ALL SELECT 'Distance', count(Distance) FROM {CLEAN}
    UNION ALL SELECT 'DepDelay', count(DepDelay) FROM {CLEAN}
    UNION ALL SELECT 'ArrDelay', count(ArrDelay) FROM {CLEAN}
    UNION ALL SELECT 'CancellationCode', count(CancellationCode) FROM {CLEAN}
    UNION ALL SELECT 'CarrierDelay', count(CarrierDelay) FROM {CLEAN}
    UNION ALL SELECT 'WeatherDelay', count(WeatherDelay) FROM {CLEAN}
    UNION ALL SELECT 'NASDelay', count(NASDelay) FROM {CLEAN}
    UNION ALL SELECT 'SecurityDelay', count(SecurityDelay) FROM {CLEAN}
    UNION ALL SELECT 'LateAircraftDelay', count(LateAircraftDelay) FROM {CLEAN}
    UNION ALL SELECT 'Delayed15', count(Delayed15) FROM {CLEAN}
)
SELECT c.column_name,
       d.data_type,
       c.non_null,
       n.n_rows - c.non_null AS missing,
       round(100.0 * (n.n_rows - c.non_null) / n.n_rows, 3) AS missing_pct
FROM counts c
JOIN workspace.information_schema.columns d
  ON d.column_name = c.column_name
 AND d.table_catalog = 'workspace'
 AND d.table_schema = 'default'
 AND d.table_name = 'airline_clean'
CROSS JOIN n
ORDER BY c.column_name
""")
display(schema)

3a. Schema description  4.68 s


column_name,data_type,non_null,missing,missing_pct
ArrDelay,DOUBLE,118503331,0,0.000
CancellationCode,STRING,10,118503321,100.000
CarrierDelay,DOUBLE,33063650,85439681,72.099
DayOfWeek,INT,118503331,0,0.000
DayofMonth,INT,118503331,0,0.000
Delayed15,BOOLEAN,118503331,0,0.000
DepDelay,DOUBLE,118503331,0,0.000
Dest,STRING,118503331,0,0.000
Distance,DOUBLE,118309103,194228,0.164
LateAircraftDelay,DOUBLE,33063650,85439681,72.099


In [0]:
%python
describe = run("3b. Summary statistics (describe)", f"""
SELECT 'Year' AS column_name, count(Year) AS n, round(avg(Year), 3) AS mean, round(stddev(Year), 3) AS std, min(Year) AS min, percentile_approx(Year, 0.01) AS p1, percentile_approx(Year, 0.25) AS p25, percentile_approx(Year, 0.5) AS p50, percentile_approx(Year, 0.75) AS p75, percentile_approx(Year, 0.95) AS p95, percentile_approx(Year, 0.99) AS p99, max(Year) AS max FROM {CLEAN}
UNION ALL SELECT 'Month', count(Month), round(avg(Month), 3), round(stddev(Month), 3), min(Month), percentile_approx(Month, 0.01), percentile_approx(Month, 0.25), percentile_approx(Month, 0.5), percentile_approx(Month, 0.75), percentile_approx(Month, 0.95), percentile_approx(Month, 0.99), max(Month) FROM {CLEAN}
UNION ALL SELECT 'DayofMonth', count(DayofMonth), round(avg(DayofMonth), 3), round(stddev(DayofMonth), 3), min(DayofMonth), percentile_approx(DayofMonth, 0.01), percentile_approx(DayofMonth, 0.25), percentile_approx(DayofMonth, 0.5), percentile_approx(DayofMonth, 0.75), percentile_approx(DayofMonth, 0.95), percentile_approx(DayofMonth, 0.99), max(DayofMonth) FROM {CLEAN}
UNION ALL SELECT 'DayOfWeek', count(DayOfWeek), round(avg(DayOfWeek), 3), round(stddev(DayOfWeek), 3), min(DayOfWeek), percentile_approx(DayOfWeek, 0.01), percentile_approx(DayOfWeek, 0.25), percentile_approx(DayOfWeek, 0.5), percentile_approx(DayOfWeek, 0.75), percentile_approx(DayOfWeek, 0.95), percentile_approx(DayOfWeek, 0.99), max(DayOfWeek) FROM {CLEAN}
UNION ALL SELECT 'Distance', count(Distance), round(avg(Distance), 3), round(stddev(Distance), 3), min(Distance), percentile_approx(Distance, 0.01), percentile_approx(Distance, 0.25), percentile_approx(Distance, 0.5), percentile_approx(Distance, 0.75), percentile_approx(Distance, 0.95), percentile_approx(Distance, 0.99), max(Distance) FROM {CLEAN}
UNION ALL SELECT 'DepDelay', count(DepDelay), round(avg(DepDelay), 3), round(stddev(DepDelay), 3), min(DepDelay), percentile_approx(DepDelay, 0.01), percentile_approx(DepDelay, 0.25), percentile_approx(DepDelay, 0.5), percentile_approx(DepDelay, 0.75), percentile_approx(DepDelay, 0.95), percentile_approx(DepDelay, 0.99), max(DepDelay) FROM {CLEAN}
UNION ALL SELECT 'ArrDelay', count(ArrDelay), round(avg(ArrDelay), 3), round(stddev(ArrDelay), 3), min(ArrDelay), percentile_approx(ArrDelay, 0.01), percentile_approx(ArrDelay, 0.25), percentile_approx(ArrDelay, 0.5), percentile_approx(ArrDelay, 0.75), percentile_approx(ArrDelay, 0.95), percentile_approx(ArrDelay, 0.99), max(ArrDelay) FROM {CLEAN}
UNION ALL SELECT 'CarrierDelay', count(CarrierDelay), round(avg(CarrierDelay), 3), round(stddev(CarrierDelay), 3), min(CarrierDelay), percentile_approx(CarrierDelay, 0.01), percentile_approx(CarrierDelay, 0.25), percentile_approx(CarrierDelay, 0.5), percentile_approx(CarrierDelay, 0.75), percentile_approx(CarrierDelay, 0.95), percentile_approx(CarrierDelay, 0.99), max(CarrierDelay) FROM {CLEAN}
UNION ALL SELECT 'WeatherDelay', count(WeatherDelay), round(avg(WeatherDelay), 3), round(stddev(WeatherDelay), 3), min(WeatherDelay), percentile_approx(WeatherDelay, 0.01), percentile_approx(WeatherDelay, 0.25), percentile_approx(WeatherDelay, 0.5), percentile_approx(WeatherDelay, 0.75), percentile_approx(WeatherDelay, 0.95), percentile_approx(WeatherDelay, 0.99), max(WeatherDelay) FROM {CLEAN}
UNION ALL SELECT 'NASDelay', count(NASDelay), round(avg(NASDelay), 3), round(stddev(NASDelay), 3), min(NASDelay), percentile_approx(NASDelay, 0.01), percentile_approx(NASDelay, 0.25), percentile_approx(NASDelay, 0.5), percentile_approx(NASDelay, 0.75), percentile_approx(NASDelay, 0.95), percentile_approx(NASDelay, 0.99), max(NASDelay) FROM {CLEAN}
UNION ALL SELECT 'SecurityDelay', count(SecurityDelay), round(avg(SecurityDelay), 3), round(stddev(SecurityDelay), 3), min(SecurityDelay), percentile_approx(SecurityDelay, 0.01), percentile_approx(SecurityDelay, 0.25), percentile_approx(SecurityDelay, 0.5), percentile_approx(SecurityDelay, 0.75), percentile_approx(SecurityDelay, 0.95), percentile_approx(SecurityDelay, 0.99), max(SecurityDelay) FROM {CLEAN}
UNION ALL SELECT 'LateAircraftDelay', count(LateAircraftDelay), round(avg(LateAircraftDelay), 3), round(stddev(LateAircraftDelay), 3), min(LateAircraftDelay), percentile_approx(LateAircraftDelay, 0.01), percentile_approx(LateAircraftDelay, 0.25), percentile_approx(LateAircraftDelay, 0.5), percentile_approx(LateAircraftDelay, 0.75), percentile_approx(LateAircraftDelay, 0.95), percentile_approx(LateAircraftDelay, 0.99), max(LateAircraftDelay) FROM {CLEAN}
""")
display(describe)

3b. Summary statistics (describe)  12.68 s


column_name,n,mean,std,min,p1,p25,p50,p75,p95,p99,max
Year,118503331,1998.633,6.244,1987.0,1987.0,1993.0,1999.0,2004.0,2008.0,2008.0,2008.0
Month,118503331,6.564,3.439,1.0,1.0,4.0,7.0,10.0,12.0,12.0,12.0
DayofMonth,118503331,15.734,8.791,1.0,1.0,8.0,16.0,23.0,29.0,31.0,31.0
DayOfWeek,118503331,3.95,1.991,1.0,1.0,2.0,4.0,6.0,7.0,7.0,7.0
Distance,118309103,708.845,554.204,0.0,79.0,309.0,550.0,946.0,1855.0,2556.0,4983.0
DepDelay,118503331,8.283,28.518,-1410.0,-10.0,-2.0,0.0,7.0,52.0,129.0,1439.0
ArrDelay,118503331,7.258,30.987,-1437.0,-27.0,-7.0,0.0,11.0,57.0,137.0,1438.0
CarrierDelay,33063650,3.807,20.086,0.0,0.0,0.0,0.0,0.0,22.0,83.0,1431.0
WeatherDelay,33063650,0.815,9.591,0.0,0.0,0.0,0.0,0.0,0.0,23.0,1429.0
NASDelay,33063650,4.244,16.858,-60.0,0.0,0.0,0.0,0.0,25.0,80.0,1392.0


In [0]:
%python
categorical = run("3c. Categorical description", f"""
WITH value_counts AS (
    SELECT 'UniqueCarrier' AS column_name, UniqueCarrier AS value, count(*) AS freq FROM {CLEAN} WHERE UniqueCarrier IS NOT NULL GROUP BY 2
    UNION ALL SELECT 'Origin', Origin, count(*) FROM {CLEAN} WHERE Origin IS NOT NULL GROUP BY 2
    UNION ALL SELECT 'Dest', Dest, count(*) FROM {CLEAN} WHERE Dest IS NOT NULL GROUP BY 2
    UNION ALL SELECT 'CancellationCode', CancellationCode, count(*) FROM {CLEAN} WHERE CancellationCode IS NOT NULL GROUP BY 2
)
SELECT column_name,
       sum(freq) AS count,
       count(*) AS n_unique,
       max_by(value, freq) AS top,
       max(freq) AS freq
FROM value_counts
GROUP BY column_name
ORDER BY column_name
""")
display(categorical)

3c. Categorical description  2.92 s


column_name,count,n_unique,top,freq
CancellationCode,10,3,A,6
Dest,118503331,343,ORD,6355471
Origin,118503331,347,ORD,6331729
UniqueCarrier,118503331,29,DL,16073976


EDA 1

In [0]:
%python
t0 = time.perf_counter()

hist = spark.sql(f"""
SELECT least(floor((ArrDelay + 60) / 5), 47) * 5 - 60 AS bin_left,
       least(floor((ArrDelay + 60) / 5), 47) * 5 - 55 AS bin_right,
       count(*) AS flights
FROM {CLEAN}
WHERE ArrDelay BETWEEN -60 AND 180
GROUP BY 1, 2
ORDER BY 1
""").toPandas()

pct = spark.sql(f"""
SELECT percentile(ArrDelay, array(0.01,0.05,0.25,0.5,0.75,0.90,0.95,0.99,0.999)) AS percentiles,
       round(avg(ArrDelay), 3) AS mean_delay,
       round(percentile(ArrDelay, 0.5), 1) AS median_delay,
       count_if(ArrDelay < -60) AS below_window,
       count_if(ArrDelay > 180) AS above_window
FROM {CLEAN}
""").toPandas()

RESULTS.append({"Stage": "4. EDA 1 delay distribution", "Seconds": round(time.perf_counter() - t0, 3)})
display(pct)
display(hist)

percentiles,mean_delay,median_delay,below_window,above_window
"List(-27.0, -18.0, -7.0, 0.0, 11.0, 32.0, 57.0, 137.0, 282.0)",7.258,0.0,3681,553029


bin_left,bin_right,flights
-60,-55,3982
-55,-50,9473
-50,-45,23415
-45,-40,58891
-40,-35,149671
-35,-30,378228
-30,-25,950108
-25,-20,2302061
-20,-15,5226842
-15,-10,10410520


EDA 2

In [0]:
%python
by_carrier = run("5. EDA 2 airline comparison", f"""
SELECT UniqueCarrier,
       count(*) AS flights,
       round(avg(ArrDelay), 3) AS mean_delay,
       round(percentile(ArrDelay, 0.5), 1) AS median_delay,
       round(100 * avg(CASE WHEN Delayed15 THEN 1.0 ELSE 0.0 END), 2) AS pct_delayed_15
FROM {CLEAN}
GROUP BY UniqueCarrier
HAVING count(*) >= greatest(1000, 0.0005 * {n_clean})
ORDER BY mean_delay
""")
display(by_carrier)

5. EDA 2 airline comparison  2.62 s


UniqueCarrier,flights,mean_delay,median_delay,pct_delayed_15
HA,258486,-0.556,-4.0,6.46
AQ,145318,1.309,-2.0,9.05
ML (1),64177,5.35,0.0,14.42
NW,9982197,5.541,-1.0,18.18
F9,333680,5.73,0.0,18.86
PA (1),294645,5.91,0.0,19.40
OO,2942603,6.113,-2.0,17.57
9E,503028,6.143,-4.0,18.77
TZ,205428,6.166,-3.0,19.05
WN,14453463,6.322,0.0,17.73


EDA 3

In [0]:
%python
by_month = run("6. EDA 3 monthly trend", f"""
SELECT Month,
       count(*) AS flights,
       round(avg(ArrDelay), 3) AS mean_delay,
       round(percentile(ArrDelay, 0.5), 1) AS median_delay,
       round(100 * avg(CASE WHEN Delayed15 THEN 1.0 ELSE 0.0 END), 2) AS pct_delayed_15
FROM {CLEAN}
GROUP BY Month
ORDER BY Month
""")
display(by_month)

6. EDA 3 monthly trend  2.12 s


Month,flights,mean_delay,median_delay,pct_delayed_15
1,9718721,8.665,1.0,22.44
2,8974597,8.111,1.0,21.55
3,10007561,7.451,0.0,20.40
4,9725754,5.438,0.0,17.25
5,9966350,5.677,-1.0,17.28
6,9841218,9.945,1.0,22.12
7,10174710,9.079,0.0,20.90
8,10250222,7.974,0.0,20.04
9,9464386,3.583,-2.0,14.77
10,10378287,4.926,0.0,16.62


EDA 4

In [0]:
%python
causes = run("7. EDA 4 delay cause breakdown", f"""
SELECT Cause, Minutes, round(100 * Minutes / sum(Minutes) OVER (), 2) AS Pct
FROM (
    SELECT 'CarrierDelay' AS Cause, coalesce(sum(CASE WHEN CarrierDelay > 0 THEN CarrierDelay END), 0) AS Minutes FROM {CLEAN}
    UNION ALL
    SELECT 'WeatherDelay', coalesce(sum(CASE WHEN WeatherDelay > 0 THEN WeatherDelay END), 0) FROM {CLEAN}
    UNION ALL
    SELECT 'NASDelay', coalesce(sum(CASE WHEN NASDelay > 0 THEN NASDelay END), 0) FROM {CLEAN}
    UNION ALL
    SELECT 'SecurityDelay', coalesce(sum(CASE WHEN SecurityDelay > 0 THEN SecurityDelay END), 0) FROM {CLEAN}
    UNION ALL
    SELECT 'LateAircraftDelay', coalesce(sum(CASE WHEN LateAircraftDelay > 0 THEN LateAircraftDelay END), 0) FROM {CLEAN}
)
ORDER BY Minutes DESC
""")
display(causes)

7. EDA 4 delay cause breakdown  2.24 s


Cause,Minutes,Pct
LateAircraftDelay,1.62671969E8,35.62
NASDelay,1.40309065E8,30.72
CarrierDelay,1.25862581E8,27.56
WeatherDelay,2.6962598E7,5.9
SecurityDelay,913344.0,0.2


In [0]:
%python
perf = pd.DataFrame(RESULTS)
total = perf["Seconds"].sum()
perf["Minutes"] = (perf["Seconds"] / 60).round(3)
perf["Pct_of_total"] = (100 * perf["Seconds"] / total).round(1)
perf.loc[len(perf)] = ["TOTAL", round(total, 3), round(total / 60, 3), 100.0]
display(perf)

Stage,Seconds,Minutes,Pct_of_total
0. Disk read benchmark (raw bytes),1.401,0.023,0.4
1. Data loading,325.259,5.421,81.9
2a. Data quality audit (pre-clean),1.335,0.022,0.3
2b. Cleaning,37.898,0.632,9.5
3a. Schema description,4.682,0.078,1.2
3b. Summary statistics (describe),12.677,0.211,3.2
3c. Categorical description,2.924,0.049,0.7
4. EDA 1 delay distribution,3.767,0.063,0.9
5. EDA 2 airline comparison,2.622,0.044,0.7
6. EDA 3 monthly trend,2.119,0.035,0.5


In [0]:
%python
display(spark.sql(f"""
SELECT 'rows loaded' AS Item, count(*) AS Value FROM {RAW}
UNION ALL SELECT 'rows kept after cleaning', count(*) FROM {CLEAN}
UNION ALL SELECT 'rows removed', (SELECT count(*) FROM {RAW}) - (SELECT count(*) FROM {CLEAN})
UNION ALL SELECT 'cancelled', count(*) FROM {RAW} WHERE Cancelled = 1
UNION ALL SELECT 'diverted', count(*) FROM {RAW} WHERE Diverted = 1
UNION ALL SELECT 'ArrDelay missing', count(*) FROM {RAW} WHERE ArrDelay IS NULL
"""))

Item,Value
rows removed,5031638
cancelled,2303324
diverted,284204
ArrDelay missing,2587529
rows loaded,123534969
rows kept after cleaning,118503331
